# What TF-IDF actually does (in 30 lines of Python)

Companion notebook to [the post on mariaa.tech](https://mariaa.tech/blog/what-tfidf-actually-does).

A six-document corpus. CountVectorizer vs TfidfVectorizer side by side. Plus a small cosine-vs-Euclidean experiment showing why document length matters.

**Estimated runtime:** ~30 seconds on a fresh Colab runtime.

## 1 · Setup

scikit-learn ships with Colab, so this is one short import block.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances

import sklearn
print("scikit-learn version:", sklearn.__version__)

## 2 · The tiny corpus

Six short documents. Two about machine learning, two about cooking, two about gardening. Picked so the topic vocabulary is obvious but stop words like `the`, `and`, `in` still dominate raw counts.

In [ ]:
corpus = [
    "The neural network learns by adjusting its weights through backpropagation.",
    "A deep neural network with many layers can model complex patterns.",
    "Heat the butter in a pan and gently fry the onions until soft.",
    "Boil the pasta in salted water until al dente, then drain.",
    "Water the tomato plants in the morning to avoid mildew on the leaves.",
    "Prune the rose bushes in early spring before new growth appears.",
]
labels = ["ML #1", "ML #2", "Cooking #1", "Cooking #2", "Gardening #1", "Gardening #2"]

for label, doc in zip(labels, corpus):
    print(f"{label:>14}: {doc}")

## 3 · CountVectorizer — top words per document by raw count

No stop-word removal. We want to see what raw counts look like — and how completely they let common words dominate.

In [ ]:
cv = CountVectorizer()
X_count = cv.fit_transform(corpus).toarray()
vocab = cv.get_feature_names_out()

def top_k(matrix, doc_idx, k=5):
    row = matrix[doc_idx]
    top_idx = np.argsort(row)[::-1][:k]
    return [(vocab[i], row[i]) for i in top_idx if row[i] > 0]

print("Top 5 words per document — by RAW COUNT\n" + "-" * 45)
for i, label in enumerate(labels):
    words = top_k(X_count, i)
    print(f"{label:>14}: " + ", ".join(f"{w}({c})" for w, c in words))

**Expected output:** "the" and similar function words dominate almost every row. The top-5 lists are mostly noise.

## 4 · TfidfVectorizer — top words per document by TF-IDF

Same corpus, same tokenization, same vocabulary. Different weights.

In [ ]:
tv = TfidfVectorizer()
X_tfidf = tv.fit_transform(corpus).toarray()
vocab_tfidf = tv.get_feature_names_out()

def top_k_tfidf(matrix, doc_idx, k=5):
    row = matrix[doc_idx]
    top_idx = np.argsort(row)[::-1][:k]
    return [(vocab_tfidf[i], row[i]) for i in top_idx if row[i] > 0]

print("Top 5 words per document — by TF-IDF\n" + "-" * 45)
for i, label in enumerate(labels):
    words = top_k_tfidf(X_tfidf, i)
    print(f"{label:>14}: " + ", ".join(f"{w}({s:.2f})" for w, s in words))

**Expected output:** the topic-specific vocabulary surfaces. `backpropagation`, `pasta`, `mildew`, `prune` — words that appear in only one document each, so they get the maximum IDF boost.

## 5 · The cosine-vs-Euclidean trap

Now add an artificially long document — the same buttery-onion sentence repeated 20 times — and check pairwise distances both ways.

In [ ]:
long_corpus = corpus + [corpus[2] * 20]   # very long cooking doc
long_labels = labels + ["Cooking #1 LONG"]

cv2 = CountVectorizer()
X = cv2.fit_transform(long_corpus).toarray()

eucl = euclidean_distances(X)
cos  = cosine_distances(X)

i_short_cook = 2
i_short_ml   = 0
i_long_cook  = len(long_corpus) - 1

print("EUCLIDEAN distance")
print(f"  short Cooking #1 <-> short ML #1   : {eucl[i_short_cook, i_short_ml]:.2f}")
print(f"  short Cooking #1 <-> long  Cooking : {eucl[i_short_cook, i_long_cook]:.2f}  <-- looks far")
print(f"  long  Cooking    <-> short ML #1   : {eucl[i_long_cook, i_short_ml]:.2f}")

print()
print("COSINE distance")
print(f"  short Cooking #1 <-> short ML #1   : {cos[i_short_cook, i_short_ml]:.2f}")
print(f"  short Cooking #1 <-> long  Cooking : {cos[i_short_cook, i_long_cook]:.2f}  <-- identical!")
print(f"  long  Cooking    <-> short ML #1   : {cos[i_long_cook, i_short_ml]:.2f}")

**Read it:** under Euclidean, the long cooking doc looks *closer to ML #1* than to its own short version — because length distorts the distance. Under cosine, the two cooking docs collapse to distance 0 (same direction in vector space). **This is why cosine is the default in NLP.**

## 6 · Your turn

- Add your own document to the corpus and re-run. Watch what TF-IDF picks up.
- Set `TfidfVectorizer(stop_words='english')` and see how the top words shift.
- Set `TfidfVectorizer(ngram_range=(1, 2))` to include bigrams — phrases like 'neural network' start surfacing.
- Try `min_df=2` to drop words that only appear once — you'll lose the rarest informative terms; that's the tradeoff.

In [ ]:
my_extra_doc = "YOUR SENTENCE HERE."
new_corpus = corpus + [my_extra_doc]

tv2 = TfidfVectorizer()
X2 = tv2.fit_transform(new_corpus).toarray()
vocab2 = tv2.get_feature_names_out()

row = X2[-1]
top_idx = np.argsort(row)[::-1][:8]
print("Top TF-IDF terms in your sentence:")
for i in top_idx:
    if row[i] > 0:
        print(f"  {vocab2[i]:<20} {row[i]:.3f}")